# 13.2 MovieLens 1M Dataset

In [ ]:
from numpy.random import randn
import numpy as np
np.random.seed(123)
import os
import matplotlib.pyplot as plt
import pandas as pd

- **GroupLens Research** cung cấp một số bộ dữ liệu đánh giá phim được thu thập từ người dùng MovieLens vào cuối những năm 1990 và đầu những năm 2000. Dữ liệu này bao gồm:
    - **Đánh giá phim**
    - **Thông tin phim (thể loại và năm sản xuất)**
    - **Dữ liệu nhân khẩu học của người dùng (tuổi, mã ZIP, giới tính, nghề nghiệp)**
- Những dữ liệu như thế này thường được sử dụng trong phát triển hệ thống gợi ý (recommendation systems) dựa trên các thuật toán học máy.
- Mặc dù trong phần này chúng ta không đi sâu vào kỹ thuật học máy, phần này sẽ hướng dẫn bạn cách xử lý và trích xuất các bộ dữ liệu như thế này để đưa chúng về dạng mà bạn cần.
- Bộ dữ liệu MovieLens 1M chứa một triệu đánh giá được thu thập từ 6.000 người dùng trên 4.000 bộ phim. Dữ liệu được lưu trong ba bảng:
    - **Ratings – bảng đánh giá**
    - **User information – bảng thông tin người dùng**
    - **Movie information – bảng thông tin phim**
> Chúng ta có thể tải từng bảng vào một pandas DataFrame bằng pandas.read_table.

In [ ]:
unames = ["user_id", "gender", "age", "occupation", "zip"]
users = pd.read_table("datasets/movielens/users.dat", sep="::",
                      header=None, names=unames, engine="python")

rnames = ["user_id", "movie_id", "rating", "timestamp"]
ratings = pd.read_table("datasets/movielens/ratings.dat", sep="::",
                        header=None, names=rnames, engine="python")

mnames = ["movie_id", "title", "genres"]
movies = pd.read_table("datasets/movielens/movies.dat", sep="::",
                       header=None, names=mnames, engine="python")

In [ ]:
# hiển thị bảng thông tin người dùng 5 dòng đầu tiên
users.head(5)

,user_id,gender,age,occupation,zip
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455


In [ ]:
# hiển thị 5 dòng đầu tiên của bảng đánh giá phim
ratings.head(5)

,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [ ]:
# hiển thị 5 dòng đầu tiên của bảng thông tin phim
movies.head(5)

,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


> Lưu ý rằng tuổi và nghề nghiệp được mã hóa dưới dạng số nguyên

- Việc phân tích dữ liệu nằm rải rác trên ba bảng không phải là một nhiệm vụ đơn giản.
- Ví dụ, nếu bạn muốn tính điểm đánh giá trung bình của một bộ phim theo giới tính và độ tuổi, thì sẽ thuận tiện hơn nếu gộp tất cả dữ liệu thành một bảng duy nhất.
- Sử dụng hàm `merge` của pandas, trước tiên chúng ta gộp bảng ratings với bảng users, sau đó gộp kết quả này với bảng movies.
    - pandas sẽ tự suy ra cột nào để dùng làm khóa gộp (merge/join) dựa trên các tên cột trùng nhau.

In [ ]:
data = pd.merge(pd.merge(ratings, users), movies)

# hiển thị toàn bộ dữ liệu đã gộp
data

,user_id,movie_id,rating,timestamp,gender,age,occupation,zip,title,genres
0,1,1193,5,978300760,F,1,10,48067,One Flew Over the Cuckoo's Nest (1975),Drama
1,1,661,3,978302109,F,1,10,48067,James and the Giant Peach (1996),Animation|Children's|Musical
2,1,914,3,978301968,F,1,10,48067,My Fair Lady (1964),Musical|Romance
3,1,3408,4,978300275,F,1,10,48067,Erin Brockovich (2000),Drama
4,1,2355,5,978824291,F,1,10,48067,"Bug's Life, A (1998)",Animation|Children's|Comedy
...,...,...,...,...,...,...,...,...,...,...
1000204,6040,1091,1,956716541,M,25,6,11106,Weekend at Bernie's (1989),Comedy
1000205,6040,1094,5,956704887,M,25,6,11106,"Crying Game, The (1992)",Drama|Romance|War
1000206,6040,562,5,956704746,M,25,6,11106,Welcome to the Dollhouse (1995),Comedy|Drama
1000207,6040,1096,4,956715648,M,25,6,11106,Sophie's Choice (1982),Drama


In [ ]:
# hiển thị hàng đầu tiên của dữ liệu đã gộp
data.iloc[0]

user_id                                            1
movie_id                                        1193
rating                                             5
timestamp                                  978300760
gender                                             F
age                                                1
occupation                                        10
zip                                            48067
title         One Flew Over the Cuckoo's Nest (1975)
genres                                         Drama
Name: 0, dtype: object

- Để tính điểm đánh giá trung bình của mỗi bộ phim theo giới tính, chúng ta có thể sử dụng phương thức pivot_table:

In [ ]:
# tạo bảng pivot để tính điểm trung bình (mean) theo tiêu đề phim và giới tính
mean_ratings = data.pivot_table("rating", index="title",
                                columns="gender", aggfunc="mean")
mean_ratings.head(5)

gender,F,M
title,,
"$1,000,000 Duck (1971)",3.375000,2.761905
'Night Mother (1986),3.388889,3.352941
'Til There Was You (1997),2.675676,2.733333
"'burbs, The (1989)",2.793478,2.962085
...And Justice for All (1979),3.828571,3.689024


- Điều này đã tạo ra một DataFrame khác chứa điểm đánh giá trung bình, với tên phim làm nhãn hàng (index) và giới tính làm nhãn cột.
- Trước tiên, cần lọc ra các bộ phim nhận được ít nhất 250 lượt đánh giá (số này mang tính tùy ý).
    - Để làm điều này, tôi nhóm dữ liệu theo cột title và dùng size() để lấy một Series cho biết kích thước của mỗi nhóm (tức số lượt đánh giá của từng phim):

In [ ]:
ratings_by_title = data.groupby("title").size()
ratings_by_title.head()
active_titles = ratings_by_title.index[ratings_by_title >= 250]
active_titles

Index([''burbs, The (1989)', '10 Things I Hate About You (1999)',
       '101 Dalmatians (1961)', '101 Dalmatians (1996)', '12 Angry Men (1957)',
       '13th Warrior, The (1999)', '2 Days in the Valley (1996)',
       '20,000 Leagues Under the Sea (1954)', '2001: A Space Odyssey (1968)',
       '2010 (1984)',
       ...
       'X-Men (2000)', 'Year of Living Dangerously (1982)',
       'Yellow Submarine (1968)', 'You've Got Mail (1998)',
       'Young Frankenstein (1974)', 'Young Guns (1988)',
       'Young Guns II (1990)', 'Young Sherlock Holmes (1985)',
       'Zero Effect (1998)', 'eXistenZ (1999)'],
      dtype='object', name='title', length=1216)

- Chỉ số (index) của các bộ phim nhận được ít nhất 250 lượt đánh giá sau đó có thể được sử dụng để chọn các hàng tương ứng từ `mean_ratings` bằng cách dùng `.loc`:

In [ ]:
mean_ratings = mean_ratings.loc[active_titles]
mean_ratings

gender,F,M
title,,
"'burbs, The (1989)",2.793478,2.962085
10 Things I Hate About You (1999),3.646552,3.311966
101 Dalmatians (1961),3.791444,3.500000
101 Dalmatians (1996),3.240000,2.911215
12 Angry Men (1957),4.184397,4.328421
...,...,...
Young Guns (1988),3.371795,3.425620
Young Guns II (1990),2.934783,2.904025
Young Sherlock Holmes (1985),3.514706,3.363344


In [ ]:
mean_ratings = mean_ratings.rename(index={"Seven Samurai (The Magnificent Seven) (Shichinin no samurai) (1954)":
                           "Seven Samurai (Shichinin no samurai) (1954)"})

- Để xem các bộ phim được nữ khán giả đánh giá cao nhất, chúng ta có thể sắp xếp theo cột `F` (giới tính nữ) theo thứ tự giảm dần:

In [ ]:
top_female_ratings = mean_ratings.sort_values("F", ascending=False)
top_female_ratings.head()

gender,F,M
title,,
24 7: Twenty Four Seven (1997),5.0,3.750000
"Gate of Heavenly Peace, The (1995)",5.0,5.000000
"Big Combo, The (1955)",5.0,3.600000
Dancemaker (1998),5.0,3.250000
Clean Slate (Coup de Torchon) (1981),5.0,3.857143


## Đo lường sự khác biệt về đánh giá

- Giả sử bạn muốn tìm những bộ phim mà có sự khác biệt lớn về đánh giá giữa nam và nữ.
    - Một cách làm là:
        - Thêm một cột vào `mean_ratings` chứa hiệu số trung bình giữa hai giới tính.
        - Sắp xếp theo cột này để tìm các bộ phim có sự khác biệt lớn nhất:

In [ ]:
mean_ratings["diff"] = mean_ratings["M"] - mean_ratings["F"]

- Sắp xếp theo cột `diff` sẽ cho ra những bộ phim có sự khác biệt điểm đánh giá lớn nhất, giúp chúng ta xem phim nào được nữ khán giả yêu thích hơn.

In [ ]:
sorted_by_diff = mean_ratings.sort_values("diff")
sorted_by_diff.head()

gender,F,M,diff
title,,,
Dirty Dancing (1987),3.790378,2.959596,-0.830782
Jumpin' Jack Flash (1986),3.254717,2.578358,-0.676359
Grease (1978),3.975265,3.367041,-0.608224
Little Women (1994),3.870588,3.321739,-0.548849
Steel Magnolias (1989),3.901734,3.365957,-0.535777


- Đảo ngược thứ tự các hàng và lấy 10 hàng đầu một lần nữa, ta sẽ có danh sách các bộ phim được nam khán giả yêu thích nhưng nữ khán giả không đánh giá cao.

In [ ]:

sorted_by_diff[::-1].head()

gender,F,M,diff
title,,,
"Good, The Bad and The Ugly, The (1966)",3.494949,4.221300,0.726351
"Kentucky Fried Movie, The (1977)",2.878788,3.555147,0.676359
Dumb & Dumber (1994),2.697987,3.336595,0.638608
"Longest Day, The (1962)",3.411765,4.031447,0.619682
"Cable Guy, The (1996)",2.250000,2.863787,0.613787


- Giả sử khi muốn tìm những bộ phim gây ra nhiều tranh cãi nhất trong số các khán giả, bất kể giới tính.
- Sự khác biệt (disagreement) có thể được đo bằng phương sai (variance) hoặc độ lệch chuẩn (standard deviation) của các lượt đánh giá.
- Để làm điều này, trước tiên ta tính độ lệch chuẩn của điểm đánh giá theo từng phim (title), sau đó lọc ra các phim hoạt động nhiều (có đủ lượt đánh giá):

In [ ]:
rating_std_by_title = data.groupby("title")["rating"].std()
rating_std_by_title = rating_std_by_title.loc[active_titles]
rating_std_by_title.head()

title
'burbs, The (1989)                   1.107760
10 Things I Hate About You (1999)    0.989815
101 Dalmatians (1961)                0.982103
101 Dalmatians (1996)                1.098717
12 Angry Men (1957)                  0.812731
Name: rating, dtype: float64

- Sau đó, chúng ta sắp xếp theo thứ tự giảm dần và chọn 10 hàng đầu, tương đương với 10 bộ phim có sự khác biệt về đánh giá lớn nhất.

In [ ]:
rating_std_by_title.sort_values(ascending=False)[:10]

title
Dumb & Dumber (1994)                     1.321333
Blair Witch Project, The (1999)          1.316368
Natural Born Killers (1994)              1.307198
Tank Girl (1995)                         1.277695
Rocky Horror Picture Show, The (1975)    1.260177
Eyes Wide Shut (1999)                    1.259624
Evita (1996)                             1.253631
Billy Madison (1995)                     1.249970
Fear and Loathing in Las Vegas (1998)    1.246408
Bicentennial Man (1999)                  1.245533
Name: rating, dtype: float64

- Có thể nhận thấy rằng thể loại phim (genres) được lưu dưới dạng chuỗi ký tự ngăn cách bằng dấu gạch dọc `|`, vì một bộ phim có thể thuộc nhiều thể loại khác nhau.
- Để giúp nhóm dữ liệu đánh giá theo thể loại, chúng ta có thể sử dụng phương thức `explode` của DataFrame.
- Trước tiên, hãy xem cách hoạt động của nó:
      - Chúng ta tách chuỗi thể loại (genres) thành danh sách các thể loại riêng biệt bằng cách dùng phương thức `str.split` trên Series.

In [ ]:
movies["genres"].head()

0     Animation|Children's|Comedy
1    Adventure|Children's|Fantasy
2                  Comedy|Romance
3                    Comedy|Drama
4                          Comedy
Name: genres, dtype: object

In [ ]:
movies["genres"].head().str.split("|")

0     [Animation, Children's, Comedy]
1    [Adventure, Children's, Fantasy]
2                   [Comedy, Romance]
3                     [Comedy, Drama]
4                            [Comedy]
Name: genres, dtype: object

In [ ]:
movies["genre"] = movies.pop("genres").str.split("|")
movies.head()

,movie_id,title,genre
0,1,Toy Story (1995),"[Animation, Children's, Comedy]"
1,2,Jumanji (1995),"[Adventure, Children's, Fantasy]"
2,3,Grumpier Old Men (1995),"[Comedy, Romance]"
3,4,Waiting to Exhale (1995),"[Comedy, Drama]"
4,5,Father of the Bride Part II (1995),[Comedy]


- Bây giờ, khi gọi `movies.explode("genre")`, ta sẽ tạo ra một DataFrame mới, trong đó mỗi phần tử bên trong danh sách thể loại của một bộ phim sẽ trở thành một hàng riêng biệt.
- Ví dụ: nếu một bộ phim được xếp vào cả hai thể loại "Hài (Comedy)" và "Lãng mạn (Romance)", thì trong kết quả sẽ có hai hàng:
    - Một hàng chứa “Comedy”,
    - Và một hàng chứa “Romance”.

In [ ]:
movies_exploded = movies.explode("genre")
movies_exploded[:10]

,movie_id,title,genre
0,1,Toy Story (1995),Animation
0,1,Toy Story (1995),Children's
0,1,Toy Story (1995),Comedy
1,2,Jumanji (1995),Adventure
1,2,Jumanji (1995),Children's
1,2,Jumanji (1995),Fantasy
2,3,Grumpier Old Men (1995),Comedy
2,3,Grumpier Old Men (1995),Romance
3,4,Waiting to Exhale (1995),Comedy
3,4,Waiting to Exhale (1995),Drama


- Bây giờ, chúng ta có thể gộp cả ba bảng dữ liệu lại với nhau và nhóm dữ liệu theo thể loại phim (genre).

In [ ]:
ratings_with_genre = pd.merge(pd.merge(movies_exploded, ratings), users)
ratings_with_genre.iloc[0]

movie_id                     1
title         Toy Story (1995)
genre                Animation
user_id                      1
rating                       5
timestamp            978824268
gender                       F
age                          1
occupation                  10
zip                      48067
Name: 0, dtype: object

In [ ]:
genre_ratings = (ratings_with_genre.groupby(["genre", "age"])
                 ["rating"].mean()
                 .unstack("age"))
genre_ratings[:10]

age,1,18,25,35,45,50,56
genre,,,,,,,
Action,3.506385,3.447097,3.453358,3.538107,3.528543,3.611333,3.610709
Adventure,3.449975,3.408525,3.443163,3.515291,3.528963,3.628163,3.649064
Animation,3.476113,3.624014,3.701228,3.740545,3.734856,3.780020,3.756233
Children's,3.241642,3.294257,3.426873,3.518423,3.527593,3.556555,3.621822
Comedy,3.497491,3.460417,3.490385,3.561984,3.591789,3.646868,3.650949
Crime,3.710170,3.668054,3.680321,3.733736,3.750661,3.810688,3.832549
Documentary,3.730769,3.865865,3.946690,3.953747,3.966521,3.908108,3.961538
Drama,3.794735,3.721930,3.726428,3.782512,3.784356,3.878415,3.933465
Fantasy,3.317647,3.353778,3.452484,3.482301,3.532468,3.581570,3.532700


In [ ]:
!head -n 10 datasets/babynames/yob1880.txt

'head' is not recognized as an internal or external command,
operable program or batch file.
